# PACE MiniGrid benchmark on Kaggle

Select **GPU** as the accelerator, enable **Internet**, and run all cells. The notebook clones the pinned trainer and generates its own MiniGrid levels.

This benchmark runs PACE `algorithm1` on one GPU with seed 0, provisional alpha 0.01, and a cap of 1,000,000 actual environment interactions. It uses the MiniGrid paper preset and includes the scored 500-level prefill. Replay collects separate PPO and scoring trajectories. Training stops before the next complete batch would exceed the cap.

Each run saves the setup and training logs, checkpoints, and `benchmark.json` under `/kaggle/working/pace/benchmark-*`. The reported training time includes prefill, JAX compilation, and checkpoint writes. Memory peaks are sampled during training. GPU memory shows total usage per visible GPU in the session, and RAM shows usage in the session cgroup when available.

In [ ]:
import os
import shutil
import subprocess
import sys
import tempfile
import zipfile
from pathlib import Path
from time import perf_counter

source_start = perf_counter()

SEED = 0
MAX_ENV_STEPS = 1_000_000
PROVISIONAL_ALPHA = 0.01  # Benchmark choice. The paper leaves this unspecified.
CHECKPOINT_INTERVAL = 10

WORK = Path('/kaggle/working')
runs = WORK / 'pace'
runs.mkdir(parents=True, exist_ok=True)
RUN = Path(tempfile.mkdtemp(prefix=f'benchmark-seed{SEED}-', dir=runs))
print(f'Benchmark output: {RUN}', flush=True)
REPO_URL = 'https://github.com/SimforianusJonathan/PACE-Research.git'
REPO_REF = 'e9086bf95d6eb0bfac9e7af55a9fdf1bb8282a43'
SOURCE = ''  # Optional path to a source ZIP or directory under /kaggle/input/.
source_id = f'local-{Path(SOURCE).stem}' if SOURCE else REPO_REF.replace('/', '-')[:12]
REPO = WORK / f'PACE-Research-{source_id}'
RUNTIME = Path('/tmp/pace-kaggle')

if not REPO.exists():
    if SOURCE:
        source = Path(SOURCE)
        if source.is_dir():
            shutil.copytree(source, REPO)
        else:
            with zipfile.ZipFile(source) as archive:
                archive.extractall(REPO)
    else:
        subprocess.run(['git', 'clone', REPO_URL, str(REPO)], check=True)
        subprocess.run(['git', 'checkout', '--detach', REPO_REF], cwd=REPO, check=True)

if not (REPO / 'scripts/setup_kaggle.sh').is_file():
    raise RuntimeError('This source lacks Kaggle support. Use a Git ref or source archive containing this notebook.')
if (REPO / '.git').exists():
    subprocess.run(['git', 'rev-parse', 'HEAD'], cwd=REPO, check=True)

# Use Kaggle's unmodified driver environment for this diagnostic.
smi = shutil.which('nvidia-smi')
if not smi or subprocess.run([smi, '-L'], check=False).returncode != 0:
    print('nvidia-smi did not list a GPU. Checking the CUDA driver directly.')

visible_device = os.environ.get('CUDA_VISIBLE_DEVICES', '0').split(',', 1)[0]
env = {**os.environ, 'PACE_RUNTIME_DIR': str(RUNTIME), 'CUDA_VISIBLE_DEVICES': visible_device,
       'JAX_PLATFORMS': 'cuda', 'XLA_PYTHON_CLIENT_PREALLOCATE': 'false',
       'MPLBACKEND': 'Agg', 'PYTHONUNBUFFERED': '1'}
# Keep mounted driver libraries while letting the wheels provide the CUDA toolkit.
paths = ['/usr/local/nvidia/lib64', *os.environ.get('LD_LIBRARY_PATH', '').split(os.pathsep)]
driver_paths = dict.fromkeys(path for path in paths if Path(path).is_absolute()
                            and Path(path).name not in {'compat', 'stubs'}
                            and (Path(path) / 'libcuda.so.1').is_file())
if driver_paths:
    env['LD_LIBRARY_PATH'] = os.pathsep.join(driver_paths)
else:
    env.pop('LD_LIBRARY_PATH', None)

driver_probe = '''
import ctypes
cuda = ctypes.CDLL('libcuda.so.1')
status = cuda.cuInit(0)
if status != 0:
    raise RuntimeError(f'CUDA driver initialization failed (cuInit={status})')
count = ctypes.c_int()
status = cuda.cuDeviceGetCount(ctypes.byref(count))
if status != 0 or count.value < 1:
    raise RuntimeError(f'No visible CUDA device (status={status}, count={count.value})')
'''
try:
    subprocess.run([sys.executable, '-c', driver_probe], env=env, check=True)
except subprocess.CalledProcessError:
    raise RuntimeError('Kaggle has no usable CUDA device for this run. Select a GPU accelerator, '
                       'then restart the session or save a new GPU-enabled version.') from None

source_seconds = perf_counter() - source_start

In [ ]:
def run_logged(command, log_path):
    log_path.parent.mkdir(parents=True, exist_ok=True)
    print(f'Log: {log_path}', flush=True)
    with log_path.open('w', encoding='utf-8') as log, subprocess.Popen(
        command, cwd=REPO, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, encoding='utf-8', errors='replace',
    ) as process:
        for line in process.stdout:
            log.write(line)
            log.flush()
            print(line, end='', flush=True)
        code = process.wait()
    if code:
        raise RuntimeError(f'{log_path.stem} failed with exit code {code}. Full log: {log_path}')

setup_start = perf_counter()
run_logged(['bash', 'scripts/setup_kaggle.sh'], RUN / 'setup.log')
PACE_PYTHON = str(RUNTIME / 'venv/bin/python')

probe = '''
import importlib.metadata as metadata
import json
import jax
import jax.numpy as jnp
from jaxued.environments import Maze
from train.pace_minigrid import MiniGridConfig
config = MiniGridConfig(provisional_alpha=0.01, protocol='algorithm1')
if not (config.uses_prefill and config.separate_replay_rollouts):
    raise RuntimeError('Use a trainer revision with scored prefill and separate replay rollouts.')
devices = jax.local_devices(backend='gpu')
assert len(devices) == 1, devices
result = jax.jit(lambda x: x @ x)(jnp.ones((16, 16))).block_until_ready()
assert result.device.platform == 'gpu' and float(result[0, 0]) == 16.0
pin = '0f8f1284677375b889e4f13a32c9617cd009f8c4'
origin = json.loads(metadata.distribution('jaxued').read_text('direct_url.json'))
assert origin['vcs_info']['commit_id'] == pin
print({'device': str(devices[0]), 'jax': jax.__version__, 'jaxued_commit': pin, 'environment': Maze.__name__})
'''
run_logged([PACE_PYTHON, '-c', probe], RUN / 'gpu-probe.log')
setup_seconds = source_seconds + perf_counter() - setup_start

In [ ]:
import threading

SAMPLE_INTERVAL_SECONDS = 2

def sample_gpu_memory():
    if smi is None:
        return {}
    try:
        query = subprocess.run(
            [smi, '--query-gpu=index,memory.used,memory.total', '--format=csv,noheader,nounits'],
            capture_output=True, text=True, check=True, timeout=5,
        )
    except (OSError, subprocess.SubprocessError):
        return {}
    readings = {}
    for line in query.stdout.splitlines():
        try:
            index, used, total = (int(field) for field in line.split(','))
        except ValueError:
            continue
        readings[str(index)] = {'used_mib': used, 'total_mib': total}
    return readings

def ram_cgroup_files():
    try:
        cgroups = Path('/proc/self/cgroup').read_text(encoding='utf-8').splitlines()
    except OSError:
        return None
    for entry in cgroups:
        _, controllers, group = entry.split(':', 2)
        if controllers == '':
            mount = Path('/sys/fs/cgroup')
            names = ('memory.current', 'memory.max')
        elif 'memory' in controllers.split(','):
            mount = Path('/sys/fs/cgroup/memory')
            names = ('memory.usage_in_bytes', 'memory.limit_in_bytes')
        else:
            continue
        root = mount / group.lstrip('/')
        usage, limit = (root / name for name in names)
        if usage.is_file():
            return usage, limit
    return None

def read_mib(path):
    try:
        value = int(path.read_text(encoding='utf-8').strip())
    except (OSError, ValueError):
        return None
    # cgroup v1 uses a value near the signed 64-bit limit for unlimited RAM.
    return round(value / (1024 ** 2), 1) if value < 2 ** 60 else None

ram_files = ram_cgroup_files()
ram_limit_mib = read_mib(ram_files[1]) if ram_files else None
gpu_samples = []
ram_samples = []
stop_monitor = threading.Event()

def monitor_memory():
    while not stop_monitor.is_set():
        gpu_samples.append(sample_gpu_memory())
        if ram_files:
            usage = read_mib(ram_files[0])
            if usage is not None:
                ram_samples.append(usage)
        stop_monitor.wait(SAMPLE_INTERVAL_SECONDS)

command = [
    PACE_PYTHON, '-m', 'train.pace_minigrid', 'train', '--device', 'gpu',
    '--protocol', 'algorithm1', '--seed', str(SEED),
    '--provisional-alpha', str(PROVISIONAL_ALPHA),
    '--budget', 'env_steps', '--max-env-steps', str(MAX_ENV_STEPS),
    '--output-dir', str(RUN), '--checkpoint-interval', str(CHECKPOINT_INTERVAL),
]

train_log = RUN / 'train.log'
monitor = threading.Thread(target=monitor_memory, daemon=True)
start = perf_counter()
monitor.start()
try:
    run_logged(command, train_log)
finally:
    training_seconds = perf_counter() - start
    stop_monitor.set()
    monitor.join()

In [ ]:
import json

records = [
    json.loads(line) for line in train_log.read_text(encoding='utf-8').splitlines()
    if line.startswith('{')
]
config = next(record['config'] for record in records if 'config' in record)
final = records[-1]
if (config['protocol'], config['budget'], config['max_env_steps']) != ('algorithm1', 'env_steps', MAX_ENV_STEPS):
    raise RuntimeError(f'Unexpected benchmark configuration: {config}')

actual = final['env_steps']
remaining = MAX_ENV_STEPS - actual
largest_batch = 2 * config['num_envs'] * config['num_steps']
if final['stop_reason'] != 'env_steps_budget' or not 0 <= remaining < largest_batch:
    raise RuntimeError(f'Unexpected benchmark stopping point: {final}')

if (final['shared_env_steps'] != 0 or
        actual != final['prefill_env_steps'] + final['ppo_env_steps'] + final['scoring_env_steps']):
    raise RuntimeError(f'Interaction counts disagree with separate replay rollouts: {final}')

checkpoint = final['checkpoint']
if not checkpoint or not Path(checkpoint).is_file():
    raise RuntimeError(f'Final checkpoint is missing: {checkpoint}')

gpu_peaks = {}
for sample in gpu_samples:
    for index, reading in sample.items():
        previous = gpu_peaks.get(index)
        if previous is None or reading['used_mib'] > previous['used_mib']:
            gpu_peaks[index] = reading

report = {
    'config': config,
    'counts': final,
    'timing': {
        'setup_seconds': round(setup_seconds, 2),
        'training_seconds': round(training_seconds, 2),
        'setup_and_training_seconds': round(setup_seconds + training_seconds, 2),
        'actual_env_steps_per_second': round(actual / training_seconds, 1),
    },
    'memory': {
        'sample_interval_seconds': SAMPLE_INTERVAL_SECONDS,
        'gpu_peak_sampled_used_mib_by_index': {
            index: reading['used_mib'] for index, reading in gpu_peaks.items()
        },
        'gpu_total_mib_by_index': {
            index: reading['total_mib'] for index, reading in gpu_peaks.items()
        },
        'cgroup_ram_peak_sampled_used_mib': max(ram_samples, default=None),
        'cgroup_ram_limit_mib': ram_limit_mib,
    },
    'trainer_ref': REPO_REF if not SOURCE else None,
}
report_path = RUN / 'benchmark.json'
report_path.write_text(json.dumps(report, indent=2) + '\n', encoding='utf-8')
print(json.dumps(report, indent=2))
print(f'Benchmark report: {report_path}')